<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/18-post-training-alignment.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **后训练与对齐** {#post-training-alignment}

预训练赋予模型广泛的预测能力，但仅靠下一词元似然，并不能规定一个部署中的助手应当如何遵循指令、拒绝不安全请求、表达不确定性，或在有用性与冗长度之间进行权衡。**后训练（post-training）**利用示范、比较、奖励信号和定向评估改变预训练模型的行为。**对齐（alignment）**则是更广泛的工程问题：使这种行为在常规与对抗性使用场景中都能可靠反映预期目标和约束。

![分阶段的后训练流水线把预测型基础模型转变为经过评估的部署策略。](assets/dl18-post-training-pipeline.svg){fig-align="center" width="78%" fig-alt="流水线从预训练模型经过监督微调和偏好优化到达已评估策略，评估结果再反馈为新数据。"}

*根据 [InstructGPT](https://arxiv.org/abs/2203.02155)、[DPO 论文](https://arxiv.org/abs/2305.18290)与 [Constitutional AI](https://arxiv.org/abs/2212.08073) 中的后训练流程综合绘制的原创教学图。*

本章使用 [NVIDIA HelpSteer2](https://huggingface.co/datasets/nvidia/HelpSteer2)，其来源见 [HelpSteer2 论文](https://arxiv.org/abs/2406.08673)。该数据集按有用性、正确性、连贯性、复杂度和冗长度对回答评分。仓库在 **CC BY 4.0** 许可下保存了官方训练集的一个确定性 240-prompt 教学子集。每个 prompt 有两个回答；加权人工分数较高者记作 $y_w$，另一条记作 $y_l$。这个约简支持可复现的 CPU 机制演示，不用于宣称排行榜成绩。

<details>
<summary><strong>Python：一次性加载 HelpSteer2，并建立防泄漏的 prompt 划分</strong></summary>

```python
import copy
import json
import math
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)


def seed_everything(seed=1818):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


data_path = Path("assets/dl18-helpsteer2-mini.jsonl")
if not data_path.exists():
    data_path = Path("ipynb/Deep-Learning/assets/dl18-helpsteer2-mini.jsonl")

rows = [json.loads(line) for line in data_path.read_text(encoding="utf-8").splitlines()]
grouped = defaultdict(list)
for row in rows:
    grouped[row["pair_id"]].append(row)


def human_score(row):
    # Verbosity is audited separately rather than rewarded by default.
    return (
        0.4 * row["helpfulness"]
        + 0.3 * row["correctness"]
        + 0.2 * row["coherence"]
        + 0.1 * row["complexity"]
    )


pairs = []
for pair_id, candidates in grouped.items():
    assert len(candidates) == 2 and candidates[0]["prompt"] == candidates[1]["prompt"]
    ordered = sorted(candidates, key=human_score, reverse=True)
    if human_score(ordered[0]) == human_score(ordered[1]):
        continue
    pairs.append({"pair_id": pair_id, "prompt": ordered[0]["prompt"], "chosen": ordered[0], "rejected": ordered[1]})

seed_everything()
random.Random(1818).shuffle(pairs)
train_pairs, val_pairs, test_pairs = pairs[:168], pairs[168:204], pairs[204:]
train_ids = {pair["pair_id"] for pair in train_pairs}
test_ids = {pair["pair_id"] for pair in test_pairs}

assert len(pairs) == 240 and len(train_pairs) == 168 and len(val_pairs) == len(test_pairs) == 36
assert train_ids.isdisjoint(test_ids)
assert all(human_score(pair["chosen"]) > human_score(pair["rejected"]) for pair in pairs)
print({"prompt split": (168, 36, 36), "responses": len(rows), "license": "CC BY 4.0"})
```

</details>

独立划分单元是 **prompt**，而不是单条回答。否则，同一指令的两个回答可能落在边界两侧，使模型或向量化器在训练时识别到测试 prompt。

### **后训练改变了什么？** {#what-post-training-changes}

基础语言模型从广泛文本中估计 $p_{\theta_0}(y\mid x)$。部署要求的是更窄、更明确的条件行为：回答用户指令、遵守策略、保留原有能力，同时避免只是在评估器看来优秀的病态策略。因此，后训练改变的是**回答上的条件分布**，而不是某个数据库中的事实内容。它可以提高期望回答的概率，但无法可靠注入模型和数据中原本不存在的知识。

一个有用的抽象是正则化策略优化：

$$
\max_\theta\;\mathbb E_{x\sim\mathcal D,\,y\sim\pi_\theta(\cdot|x)}[R(x,y)]
-\beta\,\mathbb E_x[D_{\mathrm{KL}}(\pi_\theta(\cdot|x)\|\pi_{\mathrm{ref}}(\cdot|x))].
$$

其中，$R$ 是不完美的操作性目标，$\pi_{\mathrm{ref}}$ 通常是 SFT 策略，$\beta$ 则为偏离参考策略的行为定价。较小的 $\beta$ 允许更激进的优化，也更容易利用奖励漏洞；较大的 $\beta$ 能保留参考行为，却可能阻止有益适配。SFT 通过示范实现第一次行为迁移，而 RLHF 和直接偏好方法估计或优化相对偏好。

对齐并不等于“高奖励”。目标是多维的：指令遵循、事实性、校准、安全、风格、鲁棒性、隐私和运行成本可能相互冲突。模型、奖励模型、数据收集流程与评估套件构成一个耦合系统。只提高标量代理而忽视这个系统，正是奖励投机产生的方式。

| 阶段 | 主要监督 | 直接优化对象 | 典型局限 |
|---|---|---|---|
| 预训练 | 原始词元序列 | 下一词元似然 | 部署意图是隐式的 |
| SFT | 选定的示范 | 目标回答的似然 | 模仿标注分布 |
| 奖励建模 | 回答比较 | 偏好次序的似然 | 评估器偏差与目标错设 |
| RLHF / PPO | 模型 rollout 与学习奖励 | 正则化期望奖励 | 成本高且对优化细节敏感 |
| DPO 类方法 | 固定偏好对 | 相对策略对数比 | 受离线偏好覆盖范围限制 |
| 评估与红队测试 | 保留测试与人工审查 | 诊断，而非单一训练损失 | 永远无法穷尽 |

### **监督微调** {#supervised-fine-tuning}

监督微调（SFT）教自回归模型复现高质量回答。对于 prompt $x$ 和示范 $y=(y_1,\ldots,y_T)$，仅回答区域的 SFT 最小化

$$
\mathcal L_{\mathrm{SFT}}(\theta)
=-\sum_{t=1}^{T}m_t\log \pi_\theta(y_t\mid x,y_{<t}),
$$

其中，回答目标上的 $m_t=1$，prompt 词元上的 $m_t=0$。prompt 仍作为上下文可见，但其词元通常使用忽略索引（PyTorch 中为 `-100`）。如果对 prompt 词元也计算损失，模型会浪费能力复述用户文本，而且长 prompt 样本会不成比例地主导损失。

![仅回答区域的 SFT 让 prompt 词元作为上下文可见，同时把它们从损失中屏蔽。](assets/dl18-sft-mask.svg){fig-align="center" width="76%" fig-alt="Prompt 与分隔符词元的标签为负一百，回答词元保留下一个词元标签并产生梯度。"}

把多个样本打包进同一序列可以提高加速器利用率，但注意力与标签都不得跨越样本边界。损失还应按预期统计单元归一化：按词元平均会提高长回答的权重；按样本归一化则让每条指令获得近似相同的影响。

<details>
<summary><strong>PyTorch：从一条 HelpSteer2 示范构造仅回答区域的标签</strong></summary>

```python
BOS, SEP, EOS, VOCAB_SIZE = 256, 257, 258, 259


def encode_sft_example(prompt, response, prompt_limit=96, response_limit=160):
    prompt_bytes = list(prompt.encode("utf-8")[:prompt_limit])
    response_bytes = list(response.encode("utf-8")[:response_limit])
    full = [BOS] + prompt_bytes + [SEP] + response_bytes + [EOS]
    input_ids = torch.tensor(full[:-1], dtype=torch.long)
    labels = torch.tensor(full[1:], dtype=torch.long)
    separator_index = 1 + len(prompt_bytes)
    labels[:separator_index] = -100
    return input_ids, labels, separator_index


example = train_pairs[0]
input_ids, labels, separator_index = encode_sft_example(example["prompt"], example["chosen"]["response"])
seed_everything(1819)
logits = torch.randn(len(input_ids), VOCAB_SIZE, requires_grad=True)
loss = F.cross_entropy(logits, labels, ignore_index=-100)
loss.backward()

ignored_gradient = logits.grad[:separator_index].abs().sum()
response_gradient = logits.grad[separator_index:].abs().sum()
assert ignored_gradient == 0 and response_gradient > 0
print({"input tokens": len(input_ids), "masked targets": separator_index, "response NLL": round(float(loss.detach()), 3)})
```

</details>

SFT 稳定且不可或缺，但每个 prompt 只从一个选定回答学习。它不会直接表达另一个同样流畅的回答为何略差。常见失败模式包括模板过拟合、继承示范中的冗长风格、学习率过大造成能力灾难性丢失，以及示范中缺少不确定性表达时形成的虚假自信。

### **指令数据构造** {#instruction-data-construction}

指令数据构造本身就是算法的一部分。示范并非“只是一段文本”；它编码了任务分布、评分准则、作者群体、策略边界和质量控制流程。健壮的流水线会记录来源与许可，过滤隐私或不安全材料，对语义相似的 prompt 去重，规定标注者不确定时应如何处理，并在训练前保留对抗性与分布外测试集。

![数据漏斗让原始回答依次经过来源、质量和划分检查。](assets/dl18-data-funnel.svg){fig-align="center" width="70%" fig-alt="四阶段漏斗依次执行来源检查、隐私和安全审查、准则与去重审计，以及按 prompt 的训练验证测试划分。"}

必须从多个层面理解数据平衡。只统计 prompt 数量可能掩盖某个领域的回答更长、因而产生更多损失词元；平均评分可能掩盖一个存在严重正确性问题的小群体；近重复 prompt 会夸大保留集性能。对于偏好数据，极小的分数差还会产生弱标签或不稳定标签。与其任意打破平局，不如显式保留平局，或按声明清楚的规则删除它们。

<details>
<summary><strong>Python：审计长度、评分维度、偏好间隔与划分来源</strong></summary>

```python
def split_audit(selected_pairs):
    response_lengths = [len(pair[key]["response"]) for pair in selected_pairs for key in ("chosen", "rejected")]
    score_margins = [human_score(pair["chosen"]) - human_score(pair["rejected"]) for pair in selected_pairs]
    attribute_means = {
        attribute: round(float(np.mean([pair[key][attribute] for pair in selected_pairs for key in ("chosen", "rejected")])), 3)
        for attribute in ("helpfulness", "correctness", "coherence", "complexity", "verbosity")
    }
    return {
        "prompts": len(selected_pairs),
        "median response chars": int(np.median(response_lengths)),
        "p95 response chars": int(np.percentile(response_lengths, 95)),
        "median preference margin": round(float(np.median(score_margins)), 3),
        "attribute means": attribute_means,
    }


audits = {"train": split_audit(train_pairs), "validation": split_audit(val_pairs), "test": split_audit(test_pairs)}
assert {pair["prompt"] for pair in train_pairs}.isdisjoint({pair["prompt"] for pair in test_pairs})
assert all(audits[name]["p95 response chars"] <= 2200 for name in audits)
print(audits)
```

</details>

为了让 CPU 示例可运行，本章把教学子集限制在中等回答长度，这会改变所研究的总体。因此，本章把它报告为机制数据集，而不是对完整 HelpSteer2 或真实助手流量的无偏估计。

### **偏好数据与成对反馈** {#preference-data-pairwise-feedback}

成对反馈要求标注者判断：在同一个 prompt 下，哪个回答更符合准则。与给出全局校准分数相比，这通常更容易，因为标注者只需比较两个具体候选。记录形式为 $(x,y_w,y_l)$，其中 $y_w$ 是偏好回答，$y_l$ 是被拒绝回答。

![成对反馈在一套显式准则下比较两个回答。](assets/dl18-pairwise-feedback.svg){fig-align="center" width="74%" fig-alt="一个 prompt 分支到偏好回答与拒绝回答，两者按有用性、正确性、连贯性和不确定性准则比较。"}

偏好取决于候选的生成方式。如果两个候选来自几乎相同的策略，差异可能过于细微；如果其中一个明显损坏，标签虽然容易，却无法有效刻画当前决策边界。候选次序应随机化，应记录标注者身份与不确定性，并允许平局。位置偏差、风格偏差、人口分布不匹配和准则歧义都是数据生成机制，而不是扩大规模就会自动消失的随机噪声。

HelpSteer2 提供多个独立标量维度，而不是一个普适偏好。本章为了教学构造透明的复合分数，同时把冗长度留在目标之外，以便将它作为潜在捷径单独审计。

<details>
<summary><strong>Python：度量单维评估器与多属性偏好之间的不一致</strong></summary>

```python
def proxy_judge(pair):
    # A deliberately narrow judge sees helpfulness only.
    delta = pair["chosen"]["helpfulness"] - pair["rejected"]["helpfulness"]
    if delta == 0:
        return "tie"
    return "chosen" if delta > 0 else "rejected"


proxy_labels = [proxy_judge(pair) for pair in test_pairs]
agreement = sum(label == "chosen" for label in proxy_labels) / len(proxy_labels)
tie_rate = sum(label == "tie" for label in proxy_labels) / len(proxy_labels)
assert 0 <= agreement <= 1 and 0 <= tie_rate <= 1
print({"helpfulness-only agreement": round(agreement, 3), "helpfulness ties": round(tie_rate, 3), "test pairs": len(test_pairs)})
```

</details>

不一致并不能证明任何一方一定错误：复合分数本身就包含人为权重。这个诊断说明，对齐报告必须公开准则并保留底层维度，而不能把“人类偏好”描述成脱离上下文的客观事实。

### **奖励建模** {#reward-modeling}

奖励模型把 prompt—回答对映射为标量 $r_\phi(x,y)$。在 Bradley-Terry 模型中，

$$
P_\phi(y_w\succ y_l\mid x)=\sigma\!\left(r_\phi(x,y_w)-r_\phi(x,y_l)\right),
$$

因而成对负对数似然为

$$
\mathcal L_{\mathrm{RM}}(\phi)=-\mathbb E_{(x,y_w,y_l)}\log\sigma(r_w-r_l)
=\mathbb E\,\operatorname{softplus}(-(r_w-r_l)).
$$

![Bradley-Terry 奖励模型把分数差转换为偏好概率。](assets/dl18-reward-model.svg){fig-align="center" width="74%" fig-alt="Prompt—回答特征通过标量奖励头，偏好概率等于选择回答与拒绝回答奖励差的 sigmoid。"}

能被识别的只有差值 $r_w-r_l$：给所有分数加上同一个常数不会改变结果。因此，奖励值并不是人类效用的绝对单位。除了成对准确率，还必须关注校准、子群性能、分数漂移与分布外行为。

<details>
<summary><strong>PyTorch：在固定 HelpSteer2 特征上训练紧凑的成对奖励模型</strong></summary>

```python
def candidate_text(pair, key):
    return pair["prompt"] + "\n<response>\n" + pair[key]["response"]


# Learned vocabulary and IDF statistics are fit on training text only.
vectorizer = TfidfVectorizer(max_features=1800, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vectorizer.fit([candidate_text(pair, key) for pair in train_pairs for key in ("chosen", "rejected")])


def pair_features(selected_pairs):
    chosen = vectorizer.transform([candidate_text(pair, "chosen") for pair in selected_pairs]).toarray()
    rejected = vectorizer.transform([candidate_text(pair, "rejected") for pair in selected_pairs]).toarray()
    return torch.tensor(np.stack([chosen, rejected], axis=1), dtype=torch.float32)


train_x, val_x, test_x = map(pair_features, (train_pairs, val_pairs, test_pairs))
seed_everything(1820)
reward_model = nn.Linear(train_x.shape[-1], 1)
reward_optimizer = torch.optim.AdamW(reward_model.parameters(), lr=0.04, weight_decay=2e-3)
reward_losses = []
for _ in range(350):
    scores = reward_model(train_x).squeeze(-1)
    loss = F.softplus(-(scores[:, 0] - scores[:, 1])).mean()
    reward_optimizer.zero_grad()
    loss.backward()
    reward_optimizer.step()
    reward_losses.append(float(loss.detach()))


def reward_metrics(features):
    with torch.no_grad():
        scores = reward_model(features).squeeze(-1)
        margins = scores[:, 0] - scores[:, 1]
    return {"pair accuracy": float((margins > 0).float().mean()), "mean margin": float(margins.mean())}


rm_validation = reward_metrics(val_x)
rm_test = reward_metrics(test_x)
assert reward_losses[-1] < reward_losses[0] and 0 <= rm_test["pair accuracy"] <= 1
print({"features": train_x.shape[-1], "loss": (round(reward_losses[0], 3), round(reward_losses[-1], 3)), "validation": {k: round(v, 3) for k, v in rm_validation.items()}, "test": {k: round(v, 3) for k, v in rm_test.items()}})
```

</details>

这个线性 TF-IDF 模型刻意暴露目标函数；生产级奖励模型通常复用预训练 Transformer 并连接标量头。小型保留集使准确率噪声较大，而词汇特征很容易被利用。这些限制在本章反而有教学价值，因为后续部分可以真实观察代理优化，而不是假定评估器一定正确。

### **基于人类反馈的强化学习** {#reinforcement-learning-human-feedback}

RLHF 把偏好测量与策略改进分开。人类比较回答；奖励模型学习比较规则；策略再生成新回答，并针对学习到的奖励进行优化。在 InstructGPT 风格的流水线中，SFT 既提供强初始化，也提供用于限制漂移的冻结参考策略。

![RLHF 在策略 rollout、奖励评估和带 KL 正则的 PPO 更新之间循环。](assets/dl18-rlhf-loop.svg){fig-align="center" width="76%" fig-alt="Prompt 进入策略，完整回答获得奖励与 KL 惩罚，PPO 更新策略，冻结的参考策略作为锚点。"}

常见的成形序列奖励为

$$
\widetilde R(x,y)=r_\phi(x,y)-\beta\log\frac{\pi_\theta(y|x)}{\pi_{\mathrm{ref}}(y|x)},
\qquad
\log\pi_\theta(y|x)=\sum_t\log\pi_\theta(y_t|x,y_{<t}).
$$

第一项评价完整回答，第二项惩罚在更新策略下比参考策略下显著更可能出现的序列。实现通常把逐词元 KL 惩罚分布到轨迹中，并训练价值头，把终止时奖励反向分配到各个词元。

对当前生成器而言，RLHF 是**同策略（on-policy）**的：策略移动足够远后，旧回答就不再能刻画模型当前输出。这种自适应能够暴露新失败模式，是它相对固定偏好数据的优势之一；它也使 RLHF 成本高昂，因为生成通常主导训练成本，奖励与策略版本必须同步，rollout 数据也会迅速过期。

可靠系统应记录策略与参考对数概率、原始奖励、KL 惩罚、回答长度、熵、裁剪比例、价值误差和保留人工判断。训练奖励上升但熵坍缩或 KL 快速增长是一条警报，而不是成功证明。

### **用于模型对齐的 PPO** {#ppo-model-alignment}

第 17 章把 PPO 作为通用同策略 Actor-Critic 方法介绍。在语言对齐中，状态是 prompt 加已生成前缀，动作是下一个词元，一个 episode 是完整回答。PPO 在多个梯度 epoch 中复用一次 rollout，同时裁剪概率比

$$
\rho_t(\theta)=\frac{\pi_\theta(y_t|x,y_{<t})}{\pi_{\mathrm{old}}(y_t|x,y_{<t})},
\qquad
L^{\mathrm{clip}}=\mathbb E_t\left[\min(\rho_t\hat A_t,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)\hat A_t)\right].
$$

![语言模型 PPO 把词元动作映射到终止回答分数，并使用 Critic 与 KL 成形分配信用。](assets/dl18-ppo-alignment.svg){fig-align="center" width="74%" fig-alt="词元概率生成回答，终止奖励通过 Critic 与 KL 成形向前分配，PPO 裁剪概率比的优化激励。"}

PPO 裁剪限制的是采样代理目标上的激励，并不保证所有位置上的实际 KL。显式参考 KL 与调节 $\beta$ 的控制器仍然重要。Padding mask、EOS 处理、奖励白化、优势归一化、旧对数概率的梯度分离，以及截断与真实 EOS 的区分，都是常见的静默错误来源。

<details>
<summary><strong>PyTorch：KL 正则 PPO 的双回答上下文 Bandit 抽象</strong></summary>

```python
class CandidatePolicy(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.scorer = nn.Linear(feature_dim, 1)

    def forward(self, features):
        return self.scorer(features).squeeze(-1)


def categorical_kl(logits, reference_logits):
    log_p = logits.log_softmax(-1)
    log_q = reference_logits.log_softmax(-1)
    return (log_p.exp() * (log_p - log_q)).sum(-1)


seed_everything(1821)
reference_policy = CandidatePolicy(train_x.shape[-1])
sft_optimizer = torch.optim.AdamW(reference_policy.parameters(), lr=0.025, weight_decay=1e-3)
chosen_targets = torch.zeros(len(train_x), dtype=torch.long)
for _ in range(180):
    sft_loss = F.cross_entropy(reference_policy(train_x), chosen_targets)
    sft_optimizer.zero_grad(); sft_loss.backward(); sft_optimizer.step()

for parameter in reference_policy.parameters():
    parameter.requires_grad_(False)
ppo_policy = copy.deepcopy(reference_policy)
for parameter in ppo_policy.parameters():
    parameter.requires_grad_(True)
ppo_optimizer = torch.optim.AdamW(ppo_policy.parameters(), lr=0.006, weight_decay=1e-3)

with torch.no_grad():
    train_rewards = reward_model(train_x).squeeze(-1)
    train_rewards = (train_rewards - train_rewards.mean()) / (train_rewards.std() + 1e-6)
    reference_train_logits = reference_policy(train_x)

beta, epsilon = 0.08, 0.2
ppo_log = []
for update in range(24):
    old_policy = copy.deepcopy(ppo_policy).eval()
    with torch.no_grad():
        old_logits = old_policy(train_x)
        old_distribution = torch.distributions.Categorical(logits=old_logits)
        actions = old_distribution.sample()
        old_log_prob = old_distribution.log_prob(actions)
        baseline = (old_logits.softmax(-1) * train_rewards).sum(-1)
        advantages = train_rewards.gather(1, actions[:, None]).squeeze(1) - baseline
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-6)
    for _ in range(4):
        logits = ppo_policy(train_x)
        distribution = torch.distributions.Categorical(logits=logits)
        ratio = (distribution.log_prob(actions) - old_log_prob).exp()
        surrogate = torch.minimum(ratio * advantages, ratio.clamp(1 - epsilon, 1 + epsilon) * advantages)
        kl = categorical_kl(logits, reference_train_logits).mean()
        loss = -surrogate.mean() + beta * kl - 0.002 * distribution.entropy().mean()
        ppo_optimizer.zero_grad(); loss.backward(); ppo_optimizer.step()
    ppo_log.append((float(kl.detach()), float(((ratio - 1).abs() > epsilon).float().mean())))

with torch.no_grad():
    reference_test_logits = reference_policy(test_x)
    ppo_test_logits = ppo_policy(test_x)
    ppo_preferred_probability = float(ppo_test_logits.softmax(-1)[:, 0].mean())
    ppo_test_kl = float(categorical_kl(ppo_test_logits, reference_test_logits).mean())
assert math.isfinite(ppo_test_kl) and 0 <= ppo_preferred_probability <= 1
print({"preferred probability": round(ppo_preferred_probability, 3), "test KL to reference": round(ppo_test_kl, 4), "last train clip fraction": round(ppo_log[-1][1], 3)})
```

</details>

代码把两个已记录回答视为完整动作集合。它保留 PPO 的采样、旧策略概率比、裁剪、基线与参考 KL，但并不是逐词元 RLHF，也无法暴露生成时病态行为。它的目标是在与 SFT、奖励模型相同的数据上隔离更新几何。

### **直接偏好优化** {#direct-preference-optimization}

DPO 从训练中移除了显式的“奖励模型加 RL”循环。在带 KL 正则的奖励形式下，最优策略蕴含一个与目标策略和参考策略对数比成比例的奖励。把这个关系代入 Bradley-Terry 偏好模型，可得

$$
\mathcal L_{\mathrm{DPO}}(\theta)
=-\mathbb E\log\sigma\left(\beta\left[
\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)}
-\log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}
\right]\right).
$$

![DPO 优化偏好回答相对于拒绝回答和参考策略的策略改善。](assets/dl18-dpo-log-ratio.svg){fig-align="center" width="72%" fig-alt="选择与拒绝回答的策略对数比相减、乘以 beta，再输入负对数 sigmoid。"}

方括号内是**相对改善之差**。DPO 并非简单最大化偏好回答似然，而是要求当前策略比参考策略更强地偏向 $y_w$ 而非 $y_l$。逆温度 $\beta$ 决定尺度。不同实现对它的约定并不总是一致，因此必须同时核对公式和库配置。

<details>
<summary><strong>PyTorch：在同一 HelpSteer2 偏好对上优化 DPO</strong></summary>

```python
dpo_policy = copy.deepcopy(reference_policy)
for parameter in dpo_policy.parameters():
    parameter.requires_grad_(True)
dpo_optimizer = torch.optim.AdamW(dpo_policy.parameters(), lr=0.008, weight_decay=1e-3)
with torch.no_grad():
    reference_train_log_probs = reference_policy(train_x).log_softmax(-1)

dpo_beta = 0.2
dpo_losses = []
for _ in range(220):
    policy_log_probs = dpo_policy(train_x).log_softmax(-1)
    policy_log_ratio = policy_log_probs[:, 0] - policy_log_probs[:, 1]
    reference_log_ratio = reference_train_log_probs[:, 0] - reference_train_log_probs[:, 1]
    dpo_loss = F.softplus(-dpo_beta * (policy_log_ratio - reference_log_ratio)).mean()
    dpo_optimizer.zero_grad(); dpo_loss.backward(); dpo_optimizer.step()
    dpo_losses.append(float(dpo_loss.detach()))

with torch.no_grad():
    dpo_test_logits = dpo_policy(test_x)
    dpo_preferred_probability = float(dpo_test_logits.softmax(-1)[:, 0].mean())
    dpo_test_kl = float(categorical_kl(dpo_test_logits, reference_test_logits).mean())
assert dpo_losses[-1] < dpo_losses[0] and dpo_test_kl >= 0
print({"loss": (round(dpo_losses[0], 3), round(dpo_losses[-1], 3)), "preferred probability": round(dpo_preferred_probability, 3), "test KL to reference": round(dpo_test_kl, 4)})
```

</details>

DPO 的运行流程更简单：不需要在线 rollout 服务、学习型 Critic、奖励模型在线服务或 PPO 循环。这种简单性并不会使它免受过拟合、标签噪声、长度偏差或分布外 prompt 的影响。如果不收集新偏好对，它本质上仍是离线方法。IPO、ORPO、KTO 和 robust DPO 等变体改变了假设或损失，但核心审计仍相同：数据覆盖、参考策略、正则强度和保留行为。

### **拒绝采样与 Best-of-N** {#rejection-sampling-best-of-n}

Best-of-$N$ 从提议策略采样 $N$ 个回答，分别评分，并返回最高分候选。拒绝采样微调还可以把选中的候选加入监督数据。这些方法把一部分优化转移到推理与数据选择，而不是用策略梯度改变每个词元的概率。

![Best-of-N 生成多个候选，并返回奖励模型评分最高的回答。](assets/dl18-best-of-n.svg){fig-align="center" width="72%" fig-alt="一个 prompt 产生 N 个候选，奖励模型对它们评分，argmax 阶段返回一个回答。"}

如果候选相互独立，且质量的 CDF 为 $F$，则最大值的 CDF 为 $F_{\max}(q)=F(q)^N$。更大的 $N$ 会提高评估器下最大值的期望，但在考虑批处理和缓存复用前，生成工作量约增加到 $N$ 倍。当样本缺乏多样性时，收益会饱和。更重要的是，选择极端代理分数会探索奖励模型误差可能占主导的区域。

<details>
<summary><strong>Python：比较 Best-of-N 的代理奖励与保留人工分数</strong></summary>

```python
def pair_human_scores(selected_pairs):
    return torch.tensor([[human_score(pair["chosen"]), human_score(pair["rejected"])] for pair in selected_pairs], dtype=torch.float32)


with torch.no_grad():
    proposal_probs = reference_test_logits.softmax(-1)
    test_reward_scores = reward_model(test_x).squeeze(-1)
    test_human_scores = pair_human_scores(test_pairs)


def best_of_n_metrics(n, trials=250):
    generator = torch.Generator().manual_seed(1800 + n)
    proxy_values, human_values = [], []
    for _ in range(trials):
        samples = torch.multinomial(proposal_probs, n, replacement=True, generator=generator)
        sampled_rewards = test_reward_scores.gather(1, samples)
        winner_position = sampled_rewards.argmax(1, keepdim=True)
        winner = samples.gather(1, winner_position).squeeze(1)
        proxy_values.append(test_reward_scores.gather(1, winner[:, None]).mean())
        human_values.append(test_human_scores.gather(1, winner[:, None]).mean())
    return float(torch.stack(proxy_values).mean()), float(torch.stack(human_values).mean())


best_of_n_results = {n: tuple(round(value, 3) for value in best_of_n_metrics(n)) for n in (1, 2, 4, 8)}
assert best_of_n_results[8][0] >= best_of_n_results[1][0] - 1e-6
print({"N: (RM score, human composite)": best_of_n_results})
```

</details>

由于只有两个已记录回答，这个实验隔离的是选择压力，而不是真实语言多样性。能够保证的结论只涉及被选中的**奖励模型分数**，不涉及真实人类效用。生产研究还应报告 pass@$N$、差异度、延迟、词元成本、评估器不确定性，并由人类重新评价高分样本。

### **AI 反馈与可扩展监督** {#ai-feedback-scalable-oversight}

人工比较成本高，而且很难扩展到长推理轨迹或专业领域。基于 AI 反馈的强化学习（RLAIF）使用模型评判器，在显式准则下批评、排序或修订回答。[Constitutional AI](https://arxiv.org/abs/2212.08073) 把监督式批评—修订阶段与 AI 反馈偏好阶段分开。

![Constitutional AI 使用显式准则批评和修订回答，再更新策略或评判器。](assets/dl18-rlaif.svg){fig-align="center" width="74%" fig-alt="草拟回答对照 constitution 检查，转换为修订或偏好，并在人工审计参与下用于更新。"}

可扩展流程可以询问多个评判器、随机化回答顺序、要求提供证据、允许弃权，并把高不确定性样本路由给人类。它扩展的是一套**已指定准则**，并不能证明准则完整。模型评判器可能偏爱自身风格、受到位置和冗长度影响、与策略共享盲点，或被回答中的文本操纵。

前面的“仅有用性”诊断是评判器目标错设的最小类比。更真实的评估会在隔离审计集上比较 AI 与人工标签，并按领域、语言、回答长度、安全类别和评判器置信度分层统计一致率。在简单样本上高度一致，完全可能与关键困难样本上的系统性失败同时存在。

可扩展监督还研究结果本身难以由人类直接评价的任务。分解、辩论、递归批评、过程监督和工具辅助验证都试图让潜在错误可观察。它们改变了评估器可获得的证据，却都无法取代威胁模型和独立审计。

### **在线与离线偏好学习** {#online-offline-preference-learning}

离线偏好学习在由早期策略收集的固定数据上训练。它可复现、易缓存，并适合 DPO；但其支持集也是固定的。策略发生变化后，数据中可能很少出现与新输出相似的样本。在线学习则不断从当前策略采样、获取新标签、更新并再次审计。

![离线学习继承固定覆盖范围，在线学习则形成策略—数据反馈环。](assets/dl18-online-offline.svg){fig-align="center" width="73%" fig-alt="离线面板显示固定 prompt 与偏好对；在线环把当前策略送往新反馈，再返回更新阶段。"}

在线并不自动意味着更好。新标签成本高，数据分布持续移动，回归可能污染后续收集，而且模型版本间的比较更困难。实用的混合方案先用广泛离线 SFT 与偏好建立基础，再把在线标注预算投入当前策略失败、奖励模型不确定比较和高风险切片。

<details>
<summary><strong>Python：用奖励模型不确定性安排一批优先反馈样本</strong></summary>

```python
with torch.no_grad():
    validation_reward_margin = (reward_model(val_x).squeeze(-1)[:, 0] - reward_model(val_x).squeeze(-1)[:, 1]).abs()
true_validation_margin = torch.tensor([
    human_score(pair["chosen"]) - human_score(pair["rejected"]) for pair in val_pairs
])
budget = 10
uncertain_ids = validation_reward_margin.argsort()[:budget]
random_ids = torch.randperm(len(val_pairs), generator=torch.Generator().manual_seed(1825))[:budget]
uncertainty_report = {
    "uncertainty-selected RM margin": float(validation_reward_margin[uncertain_ids].mean()),
    "random RM margin": float(validation_reward_margin[random_ids].mean()),
    "uncertainty-selected human margin": float(true_validation_margin[uncertain_ids].mean()),
    "random human margin": float(true_validation_margin[random_ids].mean()),
}
assert uncertainty_report["uncertainty-selected RM margin"] <= uncertainty_report["random RM margin"]
print({key: round(value, 3) for key, value in uncertainty_report.items()})
```

</details>

小预测间隔识别的是奖励模型认为模糊的样本，不一定是人类认为模糊的样本。把不确定性与领域覆盖、安全风险、新颖度，以及多个独立奖励模型之间的分歧结合起来，会比单一采集分数更稳健。

### **奖励投机与过度优化** {#reward-hacking-over-optimization}

当策略找到在代理指标下高分、却没有满足真实意图的行为时，就发生了奖励投机。这是 Goodhart 定律的一种形式：当带噪测量成为优化目标，优化器会寻找测量误差对自己有利的区域。[Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760) 随优化强度上升，实证区分了代理奖励与保留的“gold”奖励。

![代理奖励可能在保留人类效用达到峰值后仍继续上升。](assets/dl18-reward-overoptimization.svg){fig-align="center" width="73%" fig-alt="蓝色代理奖励曲线随优化压力持续上升，红色保留人类效用曲线先上升、达到峰值后下降。"}

捷径可能表现为冗长、过度自信、复制准则措辞、对所有请求都拒绝、奖励模型特定短语或谄媚。KL 正则能够限制策略距离，却无法保证安全：邻近策略仍可能利用局部奖励缺陷，而过强 KL 又可能保留参考策略中的不良行为。

<details>
<summary><strong>Python：压力测试一个刻意错设、对长度敏感的代理</strong></summary>

```python
with torch.no_grad():
    base_proxy = reward_model(test_x).squeeze(-1)
response_lengths = torch.tensor([
    [len(pair["chosen"]["response"]), len(pair["rejected"]["response"])] for pair in test_pairs
], dtype=torch.float32)
standardized_length = (response_lengths - response_lengths.mean()) / (response_lengths.std() + 1e-6)
true_scores = pair_human_scores(test_pairs)

reward_hacking_report = {}
for length_bonus in (0.0, 0.5, 1.0, 2.0):
    misspecified_proxy = base_proxy + length_bonus * standardized_length
    selected = misspecified_proxy.argmax(1, keepdim=True)
    reward_hacking_report[length_bonus] = {
        "optimized proxy": round(float(misspecified_proxy.gather(1, selected).mean()), 3),
        "human composite": round(float(true_scores.gather(1, selected).mean()), 3),
        "response chars": round(float(response_lengths.gather(1, selected).mean()), 1),
    }

assert reward_hacking_report[2.0]["response chars"] >= reward_hacking_report[0.0]["response chars"]
print(reward_hacking_report)
```

</details>

这是显式压力测试，并不是声称 HelpSteer2 奖励长度。关键做法是扰动评估器、针对它优化，再与保留维度比较。防御包括奖励模型集成、对抗性数据、不确定性感知惩罚、早停、参考约束、独立评判器和对极端分数输出的人工审查。它们都不能替代对真实部署分布的监控。

### **对齐代价、评估与安全** {#alignment-tax-evaluation-safety}

**对齐代价（alignment tax）**是后训练约束导致的某项原有能力、多样性、延迟或成本指标损失。它并非必然发生，也不能由一个汇总 benchmark 推断。SFT 可能通过澄清意图改善任务质量；偏好优化可能减少有用多样性；Best-of-$N$ 可能提高选中质量，同时成倍增加推理成本。正确分析对象是 Pareto 曲面，而不是一个分数。

![对齐评估分别跟踪任务质量、偏好拟合、漂移、安全、鲁棒性、效率与监督。](assets/dl18-alignment-evaluation.svg){fig-align="center" width="76%" fig-alt="七个面板构成评估账本，覆盖任务质量、偏好拟合、策略漂移、安全、鲁棒性、效率与人工监督。"}

<details>
<summary><strong>PyTorch：在共享评估账本上比较参考策略、PPO 与 DPO</strong></summary>

```python
test_lengths = response_lengths
test_human = true_scores
with torch.no_grad():
    test_rm = reward_model(test_x).squeeze(-1)


def policy_ledger(name, logits):
    probabilities = logits.softmax(-1)
    return {
        "policy": name,
        "preferred probability": float(probabilities[:, 0].mean()),
        "expected human composite": float((probabilities * test_human).sum(-1).mean()),
        "expected RM score": float((probabilities * test_rm).sum(-1).mean()),
        "KL to reference": float(categorical_kl(logits, reference_test_logits).mean()),
        "expected response chars": float((probabilities * test_lengths).sum(-1).mean()),
    }


ledgers = [
    policy_ledger("SFT reference", reference_test_logits),
    policy_ledger("PPO abstraction", ppo_test_logits),
    policy_ledger("DPO", dpo_test_logits),
]
for ledger in ledgers:
    print({key: (round(value, 3) if isinstance(value, float) else value) for key, value in ledger.items()})
assert all(math.isfinite(ledger["expected human composite"]) for ledger in ledgers)
```

</details>

这个确定性教学运行刻意允许出现令人不舒服的结果：紧凑奖励模型可以近乎完全分开训练偏好对，却在保留排序上接近随机；优化器可以提高期望学习奖励，却没有提高保留人工复合分数。这不是 PPO 与 DPO 的排名，而是说明在信任更强策略优化前，必须先证明评估器能够泛化。

这份账本有意保持不完整：HelpSteer2 的五项评分无法证明毒性、隐私、偏见、越狱抵抗、校准或下游事实正确性。安全评估需要任务专用拒绝测试、良性请求过度拒绝测试、多语言与子群切片、对抗 prompt、工具使用约束和人工升级机制。能力评估应包含未对齐基础模型或 SFT 模型以发现回归；统计报告则应包含置信区间与多种 prompt 模板。

离线分数还会遗漏系统效应。采样温度、system prompt、检索、工具、安全过滤器、量化和服务端截断都可能改变部署行为。因此，评估产物必须记录准确的模型、tokenizer、解码配置、策略文本、评估器版本与日期。

### **章节对比与总结** {#chapter-comparison-summary}

| 方法 | 训练信号 | 是否需要在线生成？ | 是否使用显式奖励模型？ | 主要优势 | 主要失败模式 |
|---|---|---:|---:|---|---|
| SFT | 目标回答词元 | 否 | 否 | 稳定、简单的指令学习 | 示范偏差，缺少负向比较 |
| 奖励建模 | 选择/拒绝回答对 | 否 | 是 | 可复用的偏好评估器 | 代理错设与分布偏移 |
| 使用 PPO 的 RLHF | 当前策略 rollout | 是 | 通常是 | 适应当前策略输出 | 系统复杂且可能利用奖励漏洞 |
| DPO | 固定偏好对 | 否 | 隐式 | 类监督学习式的简单优化 | 离线覆盖与参考策略敏感性 |
| Best-of-$N$ | 采样候选与评估器 | 推理时需要 | 通常是 | 无需策略梯度循环 | 推理成本与极端分数偏差 |
| RLAIF | 模型批评或比较 | 视情况而定 | 评判器或奖励模型 | 可扩展准则执行 | 评判器与策略共享盲点 |

实用的后训练流程是：

1. 在优化前定义预期行为与多轴评估套件。
2. 构建来源清晰的示范，并按 prompt 或对话谱系划分。
3. 使用仅回答区域标签训练 SFT，并监控能力保留。
4. 以随机化顺序收集比较，同时记录平局、不确定性和标注者元数据。
5. 按切片、校准与分布偏移验证奖励模型，而不只看训练准确率。
6. 把 DPO 作为可控离线基线；只有当前策略采样足以抵消运行成本时，才选择 PPO/RLHF。
7. 把 Best-of-$N$ 视为推理时权衡，并用独立判断重新评估极端分数。
8. 把 AI 反馈视为带人工审计的可扩展准则执行，而不是 ground truth。
9. 同时跟踪奖励、KL、熵、回答长度、能力、安全与成本。
10. 当代理奖励上升而保留性能没有改善时，停止优化或刷新数据。

最适合把后训练理解为**具备测量意识的策略塑形**。算法很重要，但它的行为边界由周围的示范、比较、参考策略、评估器、rollout 分布和部署测试共同决定。下一章将从目标函数转向系统问题：如何高效、可扩展地训练这些模型。